# 57. The Periodic Review (Base-Stock) Policy Problem

## Tier 4: The AI/ML/RL Augmentation Method (Deep Reinforcement Learning)

### Key assumptions
- Demand patterns can be non-stationary and may change over time
- System can learn from experience and adapt to changing conditions
- Historical data contains patterns that can be exploited for better decisions
- Real-time adaptation provides value over static policies

### Approach (step-by-step)
1. **Environment Modeling**: Create a Markov Decision Process for inventory management
2. **State Representation**: Design comprehensive state space (inventory, demand history, etc.)
3. **Action Space**: Define continuous order quantity decisions
4. **Reward Design**: Create cost-based reward function for learning
5. **Deep Q-Network**: Implement DQN with experience replay and target networks
6. **Training**: Train agent through episodic interaction with environment
7. **Evaluation**: Test learned policy against traditional methods

### What to look for in the results
- Learning curves showing cost improvement over episodes
- Adaptive behavior in response to changing demand patterns
- Performance comparison with traditional base-stock policies
- Policy analysis showing learned decision patterns

### Concrete example (from the source)
Expected DQN performance:
- **Training Progress**: Cost reduction from $2,847 to $2,187 over 1000 episodes
- **Final Performance**: $2,187/period cost, 96.4% service level
- **Improvement vs Traditional**: 4.5% cost reduction, 1.5% service improvement
- **Adaptive Features**: Dynamic ordering based on recent patterns

### Why this Tier exists vs Tiers 1-3
Deep RL addresses fundamental limitations of static optimization approaches:
- **Non-Stationarity**: Can adapt to changing demand patterns over time
- **Learning from Data**: Improves performance through experience
- **Complex Patterns**: Captures non-linear relationships missed by analytical models
- **Real-time Adaptation**: Continuously adjusts policy based on recent observations
- **Uncertainty Handling**: Learns to operate effectively under uncertainty

### Pros / Cons vs Tiers 1-3
**Pros:**
- Adapts to changing conditions and non-stationary demand
- Learns complex patterns from historical data
- Provides continuous improvement through experience
- Handles non-linear relationships and complex state spaces
- Can outperform static policies in dynamic environments
- Robust to model misspecification

**Cons:**
- Requires significant training data and computational resources
- Performance may be unstable during training
- Less interpretable than analytical methods
- May overfit to training data patterns
- Requires careful hyperparameter tuning
- No theoretical optimality guarantees

### When to use this Tier
- When demand patterns are non-stationary or seasonal
- When you have abundant historical data for training
- When adaptive performance is more valuable than static optimality
- When traditional models fail to capture complex patterns
- When real-time adaptation provides competitive advantage
- When you can afford computational training costs

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm
from collections import deque
import random
import warnings
warnings.filterwarnings('ignore')

# Set style for professional visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")

In [ ]:
class InventoryEnvironment:
    """
    Inventory Management Environment for Deep Reinforcement Learning
    
    This environment simulates a periodic review inventory system where
    an agent learns to make optimal ordering decisions through interaction.
    """
    
    def __init__(self, demand_mean=12000, demand_std=2400, lead_time=0.5,
                 holding_cost=0.15, ordering_cost=75, stockout_cost=8,
                 max_inventory=50000, review_period=1.0, demand_volatility=0.1):
        """
        Initialize the inventory environment.
        """
        # Problem parameters
        self.demand_mean = demand_mean
        self.demand_std = demand_std
        self.lead_time = lead_time
        self.holding_cost = holding_cost
        self.ordering_cost = ordering_cost
        self.stockout_cost = stockout_cost
        self.max_inventory = max_inventory
        self.review_period = review_period
        self.demand_volatility = demand_volatility
        
        # State variables
        self.current_inventory = 0
        self.pipeline_orders = deque(maxlen=int(lead_time * 10))
        self.demand_history = deque([demand_mean] * 10, maxlen=10)
        self.time_step = 0
        self.episode_step = 0
        
        # Demand pattern parameters (for non-stationarity)
        self.demand_trend = 0
        self.seasonal_phase = 0
        
        # State space dimension
        self.state_dim = 13  # inventory + pipeline + time + demand history (10)
        
        print(f"Inventory Environment initialized:")
        print(f"  Demand: N({self.demand_mean:,}, {self.demand_std:,}²)")
        print(f"  Lead time: {self.lead_time} weeks, Review period: {self.review_period} weeks")
        print(f"  State dimension: {self.state_dim}")
    
    def reset(self):
        """
        Reset environment to initial state for new episode.
        """
        # Initialize inventory at a reasonable level
        initial_inventory = self.demand_mean * (self.review_period + self.lead_time)
        self.current_inventory = initial_inventory
        
        # Clear pipeline orders
        self.pipeline_orders.clear()
        
        # Reset demand history
        self.demand_history = deque([self.demand_mean] * 10, maxlen=10)
        
        # Reset time counters
        self.time_step = 0
        self.episode_step = 0
        
        # Randomize demand pattern for variety
        self.demand_trend = random.uniform(-0.02, 0.02)  # Small trend
        self.seasonal_phase = random.uniform(0, 2 * np.pi)  # Random phase
        
        return self._get_state()
    
    def _get_state(self):
        """
        Get current state representation.
        State includes: inventory, pipeline, time fraction, demand history
        """
        # Normalize inventory and pipeline
        norm_inventory = self.current_inventory / self.max_inventory
        norm_pipeline = sum(self.pipeline_orders) / self.max_inventory
        
        # Time fraction within review period
        time_fraction = (self.episode_step % int(self.review_period * 10)) / (self.review_period * 10)
        
        # Normalize demand history
        avg_demand = np.mean(self.demand_history) if self.demand_history else self.demand_mean
        norm_demand_history = [d / (2 * avg_demand) for d in self.demand_history]
        
        # Combine all state components
        state = [norm_inventory, norm_pipeline, time_fraction] + norm_demand_history
        
        return np.array(state, dtype=np.float32)
    
    def _generate_demand(self):
        """
        Generate demand with non-stationary patterns.
        """
        # Base demand with trend and seasonality
        trend_effect = 1 + self.demand_trend * self.episode_step / 520  # Yearly cycle
        seasonal_effect = 1 + 0.1 * np.sin(self.seasonal_phase + self.episode_step / 26)
        
        # Adjusted demand parameters
        current_mean = self.demand_mean * trend_effect * seasonal_effect
        
        # Add some volatility
        volatility_factor = 1 + self.demand_volatility * np.sin(self.episode_step / 13)
        current_std = self.demand_std * volatility_factor
        
        # Generate demand
        demand = max(0, np.random.normal(current_mean, current_std))
        
        return demand
    
    def step(self, action):
        """
        Execute one time step in the environment.
        
        Args:
            action: Continuous action in [0, 1] representing order quantity ratio
            
        Returns:
            next_state, reward, done, info
        """
        # Decode action to order quantity
        max_order = self.max_inventory * 0.5  # Conservative max order
        order_quantity = action * max_order
        
        # Receive pipeline orders
        if len(self.pipeline_orders) > 0:
            received_order = self.pipeline_orders.popleft()
            self.current_inventory += received_order
        
        # Generate and satisfy demand
        demand = self._generate_demand()
        self.demand_history.append(demand)
        
        shortage = max(0, demand - self.current_inventory)
        self.current_inventory = max(0, self.current_inventory - demand)
        
        # Calculate costs
        holding_cost = self.holding_cost * self.current_inventory
        ordering_cost = self.ordering_cost if order_quantity > 0 else 0
        stockout_cost = self.stockout_cost * shortage
        total_cost = holding_cost + ordering_cost + stockout_cost
        
        # Update pipeline with new order
        if len(self.pipeline_orders) == self.pipeline_orders.maxlen:
            # Pipeline is full, drop oldest order (simplified)
            self.pipeline_orders.popleft()
        self.pipeline_orders.append(order_quantity)
        
        # Update time
        self.episode_step += 1
        self.time_step += 1
        
        # Check if episode is done (52 weeks = 1 year)
        done = self.episode_step >= 520
        
        # Reward is negative total cost (we want to minimize cost)
        reward = -total_cost
        
        # Info dictionary
        info = {
            'total_cost': total_cost,
            'holding_cost': holding_cost,
            'ordering_cost': ordering_cost,
            'stockout_cost': stockout_cost,
            'demand': demand,
            'shortage': shortage,
            'order_quantity': order_quantity,
            'inventory_level': self.current_inventory
        }
        
        return self._get_state(), reward, done, info

print("InventoryEnvironment class defined successfully!")

In [ ]:
class DQNNetwork:
    """
    Deep Q-Network implementation for inventory management.
    
    This neural network approximates the Q-function for the inventory management problem.
    It maps states to action values, allowing the agent to learn optimal policies.
    """
    
    def __init__(self, state_dim, action_dim, hidden_dim=64, learning_rate=0.001):
        """
        Initialize the Deep Q-Network.
        
        Args:
            state_dim: Dimension of the state space
            action_dim: Number of possible actions (order quantities)
            hidden_dim: Number of neurons in hidden layers
            learning_rate: Learning rate for optimization
        """
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.hidden_dim = hidden_dim
        self.learning_rate = learning_rate
        
        # Initialize weights and biases
        # Input layer -> Hidden layer
        self.weights = {
            'W0': np.random.randn(state_dim, hidden_dim) * 0.1,
            'W1': np.random.randn(hidden_dim, hidden_dim) * 0.1,
            'W2': np.random.randn(hidden_dim, action_dim) * 0.1
        }
        
        self.biases = {
            'b0': np.zeros(hidden_dim),
            'b1': np.zeros(hidden_dim),
            'b2': np.zeros(action_dim)
        }
        
        # Initialize gradients (always available)
        self.grad_weights = {
            'W0': np.zeros_like(self.weights['W0']),
            'W1': np.zeros_like(self.weights['W1']),
            'W2': np.zeros_like(self.weights['W2'])
        }
        
        self.grad_biases = {
            'b0': np.zeros_like(self.biases['b0']),
            'b1': np.zeros_like(self.biases['b1']),
            'b2': np.zeros_like(self.biases['b2'])
        }
        
        print(f"DQN Network initialized:")
        print(f"  State dimension: {state_dim}")
        print(f"  Action dimension: {action_dim}")
        print(f"  Hidden dimension: {hidden_dim}")
        print(f"  Learning rate: {learning_rate}")
    
    def relu(self, x):
        """ReLU activation function."""
        return np.maximum(0, x)
    
    def relu_derivative(self, x):
        """Derivative of ReLU activation function."""
        return (x > 0).astype(float)
    
    def forward(self, state):
        """
        Forward pass through the network.
        
        Args:
            state: Current state vector
            
        Returns:
            Q-values for all actions
        """
        # Store intermediate values for backpropagation
        self.cache = {}
        
        # Input layer -> Hidden layer 1
        z0 = np.dot(state, self.weights['W0']) + self.biases['b0']
        a0 = self.relu(z0)
        self.cache['z0'] = z0
        self.cache['a0'] = a0
        
        # Hidden layer 1 -> Hidden layer 2
        z1 = np.dot(a0, self.weights['W1']) + self.biases['b1']
        a1 = self.relu(z1)
        self.cache['z1'] = z1
        self.cache['a1'] = a1
        
        # Hidden layer 2 -> Output layer
        z2 = np.dot(a1, self.weights['W2']) + self.biases['b2']
        self.cache['z2'] = z2
        
        return z2
    
    def backward(self, state, target_q_values):
        """
        Backward pass to compute gradients.
        
        Args:
            state: Input state
            target_q_values: Target Q-values
        """
        # Forward pass to get cache
        q_values = self.forward(state)
        
        # Compute loss gradient (MSE loss)
        loss_gradient = 2 * (q_values - target_q_values)
        
        # Backpropagate through output layer
        dz2 = loss_gradient
        self.grad_weights['W2'] = np.outer(self.cache['a1'], dz2)
        self.grad_biases['b2'] = dz2
        
        # Backpropagate through hidden layer 2
        da1 = np.dot(dz2, self.weights['W2'].T)
        dz1 = da1 * self.relu_derivative(self.cache['z1'])
        self.grad_weights['W1'] = np.outer(self.cache['a0'], dz1)
        self.grad_biases['b1'] = dz1
        
        # Backpropagate through hidden layer 1
        da0 = np.dot(dz1, self.weights['W1'].T)
        dz0 = da0 * self.relu_derivative(self.cache['z0'])
        self.grad_weights['W0'] = np.outer(state, dz0)
        self.grad_biases['b0'] = dz0
    
    def update_weights(self):
        """Update weights using gradient descent."""
        for key in self.weights:
            self.weights[key] -= self.learning_rate * self.grad_weights[key]
            self.biases[key] -= self.learning_rate * self.grad_biases[key]
    
    def get_action(self, state, epsilon=0.0):
        """
        Get action using epsilon-greedy policy.
        
        Args:
            state: Current state
            epsilon: Exploration probability
            
        Returns:
            Selected action
        """
        if np.random.random() < epsilon:
            # Random exploration
            return np.random.randint(0, self.action_dim)
        else:
            # Greedy action
            q_values = self.forward(state)
            return np.argmax(q_values)
    
    def copy_weights_from(self, other_network):
        """Copy weights from another network."""
        for key in self.weights:
            self.weights[key] = other_network.weights[key].copy()
            self.biases[key] = other_network.biases[key].copy()

print("DQNNetwork class defined successfully!")

In [ ]:
class DQNAgent:
    """
    Deep Q-Network Agent for Inventory Management
    
    Implements DQN with experience replay and target network.
    """
    
    def __init__(self, state_dim, action_dim=20, learning_rate=0.001, gamma=0.99,
                 epsilon=1.0, epsilon_decay=0.995, epsilon_min=0.05, memory_size=10000):
        """
        Initialize DQN agent.
        """
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.learning_rate = learning_rate
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min
        self.batch_size = 32
        
        # Neural networks
        self.q_network = DQNNetwork(state_dim, action_dim, learning_rate=learning_rate)
        self.target_network = DQNNetwork(state_dim, action_dim, learning_rate=learning_rate)
        
        # Initialize target network
        self.update_target_network()
        
        # Experience replay buffer
        self.memory = deque(maxlen=memory_size)
        
        # Training statistics
        self.training_step = 0
        
        print(f"DQN Agent initialized:")
        print(f"  State dim: {state_dim}, Action dim: {action_dim}")
        print(f"  Learning rate: {learning_rate}, Gamma: {gamma}")
        print(f"  Epsilon: {epsilon} → {epsilon_min} (decay: {epsilon_decay})")
        print(f"  Memory size: {memory_size}, Batch size: {self.batch_size}")
    
    def update_target_network(self):
        """
        Copy weights from Q-network to target network.
        """
        for key in ['W0', 'W1', 'W2']:
            self.target_network.weights[key] = self.q_network.weights[key].copy()
        for key in ['b0', 'b1', 'b2']:
            self.target_network.biases[key] = self.q_network.biases[key].copy()
    
    def remember(self, state, action, reward, next_state, done):
        """
        Store experience in replay buffer.
        """
        self.memory.append((state, action, reward, next_state, done))
    
    def act(self, state, training=True):
        """
        Choose action using epsilon-greedy policy.
        
        Args:
            state: Current state
            training: Whether to use exploration
            
        Returns:
            Continuous action in [0, 1]
        """
        if training and random.random() <= self.epsilon:
            # Explore: random continuous action
            return random.random()
        
        # Exploit: choose best action
        q_values = self.q_network.forward(state)
        best_action = np.argmax(q_values)
        
        # Convert discrete action to continuous
        return best_action / (self.action_dim - 1)
    
    def replay(self):
        """
        Train the network using experience replay.
        """
        if len(self.memory) < self.batch_size:
            return
        
        # Sample random batch from memory
        batch = random.sample(self.memory, self.batch_size)
        
        # Train on each experience individually (simplified)
        for state, action, reward, next_state, done in batch:
            # Convert continuous action to discrete
            discrete_action = int(action * (self.action_dim - 1))
            discrete_action = np.clip(discrete_action, 0, self.action_dim - 1)
            
            # Calculate target Q-value
            if done:
                target_q = reward
            else:
                next_q_values = self.target_network.forward(next_state)
                target_q = reward + self.gamma * np.max(next_q_values)
            
            # Create target Q-values array
            current_q_values = self.q_network.forward(state)
            target_q_values = current_q_values.copy()
            target_q_values[discrete_action] = target_q
            
            # Train network
            self.q_network.backward(state, target_q_values)
            self.q_network.update_weights()
        
        # Decay epsilon
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay
        
        self.training_step += 1
        
        # Update target network periodically
        if self.training_step % 100 == 0:
            self.update_target_network()

print("DQNAgent class defined successfully!")

In [ ]:
# Initialize environment and agent
env = InventoryEnvironment(
    demand_mean=12000,
    demand_std=2400,
    lead_time=0.5,
    holding_cost=0.15,
    ordering_cost=75,
    stockout_cost=8,
    max_inventory=50000,
    review_period=1.0,
    demand_volatility=0.1
)

agent = DQNAgent(
    state_dim=env.state_dim,
    action_dim=20,
    learning_rate=0.001,
    gamma=0.99,
    epsilon=1.0,
    epsilon_decay=0.995,
    epsilon_min=0.05,
    memory_size=10000
)

print("Environment and agent initialized successfully!")

In [ ]:
def train_agent(agent, env, num_episodes=1000, max_steps_per_episode=520, verbose=True):
    """
    Train the DQN agent.
    
    Args:
        agent: DQN agent
        env: Inventory environment
        num_episodes: Number of training episodes
        max_steps_per_episode: Maximum steps per episode
        verbose: Whether to print progress
        
    Returns:
        Training history
    """
    training_history = []
    
    print(f"Starting training for {num_episodes} episodes...")
    print("-" * 60)
    
    for episode in range(num_episodes):
        state = env.reset()
        total_reward = 0
        total_cost = 0
        total_shortage = 0
        steps = 0
        
        while steps < max_steps_per_episode:
            # Choose action
            action = agent.act(state, training=True)
            
            # Take step
            next_state, reward, done, info = env.step(action)
            
            # Store experience
            agent.remember(state, action, reward, next_state, done)
            
            # Update statistics
            total_reward += reward
            total_cost += info['total_cost']
            total_shortage += info['shortage']
            
            # Train agent
            if len(agent.memory) > agent.batch_size:
                agent.replay()
            
            state = next_state
            steps += 1
            
            if done:
                break
        
        # Calculate episode metrics
        avg_cost_per_step = total_cost / steps if steps > 0 else 0
        service_level = 1 - (total_shortage / (steps * env.demand_mean)) if steps > 0 else 0
        avg_inventory = info['inventory_level']  # Final inventory as approximation
        
        # Store episode data
        episode_data = {
            'episode': episode,
            'total_reward': total_reward,
            'avg_cost': avg_cost_per_step,
            'service_level': service_level,
            'avg_inventory': avg_inventory,
            'epsilon': agent.epsilon,
            'steps': steps
        }
        training_history.append(episode_data)
        
        # Print progress
        if verbose and (episode + 1) % 100 == 0:
            print(f"Episode {episode + 1:4d}: "
                  f"Reward = {total_reward:8.0f}, "
                  f"Cost = ${avg_cost_per_step:7.2f}, "
                  f"Service = {service_level:5.3f}, "
                  f"Epsilon = {agent.epsilon:5.3f}")
    
    print("-" * 60)
    print("Training completed!")
    
    return training_history

def test_agent(agent, env, num_episodes=10, max_steps_per_episode=520):
    """
    Test the trained agent.
    
    Args:
        agent: Trained DQN agent
        env: Inventory environment
        num_episodes: Number of test episodes
        max_steps_per_episode: Maximum steps per episode
        
    Returns:
        Test results
    """
    test_results = []
    
    # Set epsilon to 0 for greedy policy
    original_epsilon = agent.epsilon
    agent.epsilon = 0
    
    print(f"\nTesting agent for {num_episodes} episodes...")
    
    for episode in range(num_episodes):
        state = env.reset()
        total_reward = 0
        total_cost = 0
        total_shortage = 0
        inventory_levels = []
        order_quantities = []
        demands = []
        steps = 0
        
        while steps < max_steps_per_episode:
            # Choose action (greedy)
            action = agent.act(state, training=False)
            
            # Take step
            next_state, reward, done, info = env.step(action)
            
            # Collect data
            total_reward += reward
            total_cost += info['total_cost']
            total_shortage += info['shortage']
            inventory_levels.append(info['inventory_level'])
            order_quantities.append(info['order_quantity'])
            demands.append(info['demand'])
            
            state = next_state
            steps += 1
            
            if done:
                break
        
        # Calculate metrics
        avg_cost_per_step = total_cost / steps if steps > 0 else 0
        service_level = 1 - (total_shortage / sum(demands)) if sum(demands) > 0 else 0
        avg_inventory = np.mean(inventory_levels) if inventory_levels else 0
        total_orders = sum(1 for q in order_quantities if q > 0)
        avg_order_size = np.mean([q for q in order_quantities if q > 0]) if total_orders > 0 else 0
        
        episode_result = {
            'episode': episode,
            'avg_cost': avg_cost_per_step,
            'service_level': service_level,
            'avg_inventory': avg_inventory,
            'total_orders': total_orders,
            'avg_order_size': avg_order_size,
            'total_reward': total_reward
        }
        test_results.append(episode_result)
    
    # Restore original epsilon
    agent.epsilon = original_epsilon
    
    return test_results

print("Training and testing functions defined successfully!")

In [ ]:
# Train the DQN agent
# Note: This may take a few minutes to complete
# Reduced parameters for faster execution
training_history = train_agent(agent, env, num_episodes=500, verbose=True)  # Reduced from 1000

In [ ]:
# Test the trained agent
test_results = test_agent(agent, env, num_episodes=10)
test_df = pd.DataFrame(test_results)

print("\n" + "="*60)
print("TEST RESULTS SUMMARY")
print("="*60)

# Calculate test statistics
test_stats = {
    'avg_cost': test_df['avg_cost'].mean(),
    'cost_std': test_df['avg_cost'].std(),
    'avg_service': test_df['service_level'].mean(),
    'service_std': test_df['service_level'].std(),
    'avg_inventory': test_df['avg_inventory'].mean(),
    'inventory_std': test_df['avg_inventory'].std(),
    'avg_orders': test_df['total_orders'].mean(),
    'avg_order_size': test_df['avg_order_size'].mean()
}

print(f"Average Cost: ${test_stats['avg_cost']:.2f} ± ${test_stats['cost_std']:.2f}/period")
print(f"Average Service Level: {test_stats['avg_service']:.3f} ± {test_stats['service_std']:.3f} ({test_stats['avg_service']*100:.1f}% ± {test_stats['service_std']*100:.1f}%)")
print(f"Average Inventory: {test_stats['avg_inventory']:.0f} ± {test_stats['inventory_std']:.0f} units")
print(f"Average Orders per Episode: {test_stats['avg_orders']:.1f}")
print(f"Average Order Size: {test_stats['avg_order_size']:.0f} units")

# Compare with traditional methods
print("\n" + "="*50)
print("COMPARISON WITH TRADITIONAL METHODS")
print("="*50)

# Traditional base-stock policy results (from Tiers 1-2)
traditional_cost = 2284.73  # Tier 2 result
traditional_service = 0.942  # Tier 2 result

cost_improvement_vs_traditional = (traditional_cost - test_stats['avg_cost']) / traditional_cost * 100
service_improvement_vs_traditional = (test_stats['avg_service'] - traditional_service) / traditional_service * 100

print(f"Traditional Base-Stock Policy:")
print(f"  Cost: ${traditional_cost:.2f}/period")
print(f"  Service Level: {traditional_service:.3f} ({traditional_service*100:.1f}%)")

print(f"\nTrained DQN Agent:")
print(f"  Cost: ${test_stats['avg_cost']:.2f}/period")
print(f"  Service Level: {test_stats['avg_service']:.3f} ({test_stats['avg_service']*100:.1f}%)")

print(f"\nImprovement vs Traditional:")
print(f"  Cost Reduction: {cost_improvement_vs_traditional:.1f}%")
print(f"  Service Improvement: {service_improvement_vs_traditional:.1f}%")

if cost_improvement_vs_traditional > 0 and service_improvement_vs_traditional > 0:
    print("\n✓ DQN agent OUTPERFORMS traditional methods in both cost and service!")
elif cost_improvement_vs_traditional > 0:
    print("\n✓ DQN agent achieves COST REDUCTION vs traditional methods")
elif service_improvement_vs_traditional > 0:
    print("\n✓ DQN agent achieves SERVICE IMPROVEMENT vs traditional methods")
else:
    print("\n⚠ DQN agent needs more training to match traditional performance")

In [ ]:
# Create comprehensive visualization
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Deep Reinforcement Learning for Base-Stock Policy', fontsize=16, fontweight='bold')

# Plot 1: Training Progress - Cost
window_size = 50
training_df['cost_ma'] = training_df['avg_cost'].rolling(window=window_size).mean()
ax1.plot(training_df['episode'], training_df['avg_cost'], 'b-', alpha=0.3, linewidth=0.5, label='Raw')
ax1.plot(training_df['episode'], training_df['cost_ma'], 'b-', linewidth=2, label=f'MA({window_size})')
ax1.axhline(y=traditional_cost, color='red', linestyle='--', alpha=0.7, label=f'Traditional: ${traditional_cost:.0f}')
ax1.set_xlabel('Episode')
ax1.set_ylabel('Average Cost ($/period)')
ax1.set_title('Training Progress: Cost Reduction')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Training Progress - Service Level
training_df['service_ma'] = training_df['service_level'].rolling(window=window_size).mean()
ax2.plot(training_df['episode'], training_df['service_level'], 'g-', alpha=0.3, linewidth=0.5, label='Raw')
ax2.plot(training_df['episode'], training_df['service_ma'], 'g-', linewidth=2, label=f'MA({window_size})')
ax2.axhline(y=traditional_service, color='red', linestyle='--', alpha=0.7, label=f'Traditional: {traditional_service:.3f}')
ax2.set_xlabel('Episode')
ax2.set_ylabel('Service Level')
ax2.set_title('Training Progress: Service Level')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Epsilon Decay
ax3.plot(training_df['episode'], training_df['epsilon'], 'r-', linewidth=2)
ax3.set_xlabel('Episode')
ax3.set_ylabel('Epsilon (Exploration Rate)')
ax3.set_title('Exploration-Exploitation Balance')
ax3.grid(True, alpha=0.3)

# Plot 4: Performance Comparison
methods = ['Traditional', 'DQN Agent']
costs = [traditional_cost, test_stats['avg_cost']]
services = [traditional_service, test_stats['avg_service']]

x = np.arange(len(methods))
width = 0.35

# Normalize for comparison
cost_norm = [c / max(costs) for c in costs]
service_norm = [s / max(services) for s in services]

ax4.bar(x - width/2, cost_norm, width, label='Cost (normalized)', alpha=0.8)
ax4.bar(x + width/2, service_norm, width, label='Service (normalized)', alpha=0.8)

ax4.set_xlabel('Method')
ax4.set_ylabel('Normalized Performance (0-1)')
ax4.set_title('DQN vs Traditional Performance')
ax4.set_xticks(x)
ax4.set_xticklabels(methods)
ax4.legend()
ax4.grid(True, alpha=0.3)

# Add actual values as text
for i, (cost, service) in enumerate(zip(costs, services)):
    ax4.text(i - width/2, cost_norm[i] + 0.02, f'${cost:.0f}', ha='center', va='bottom')
    ax4.text(i + width/2, service_norm[i] + 0.02, f'{service:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print("Visualization complete! Key insights:")
print(f"1. DQN agent learned to reduce cost from ${initial_performance['avg_cost']:.0f} to ${final_performance['avg_cost']:.0f}")
print(f"2. Service level improved from {initial_performance['service_level']:.3f} to {final_performance['service_level']:.3f}")
print(f"3. Epsilon decay balanced exploration and exploitation")
print(f"4. Final performance: {cost_improvement_vs_traditional:.1f}% cost improvement, {service_improvement_vs_traditional:.1f}% service improvement")

In [ ]:
# Policy Analysis
print("POLICY ANALYSIS")
print("="*50)

# Analyze learned policy by testing with different states
def analyze_policy(agent, env, num_test_states=100):
    """
    Analyze the learned policy behavior.
    """
    policy_data = []
    
    for _ in range(num_test_states):
        # Generate random state
        state = env.reset()
        
        # Get Q-values
        q_values = agent.q_network.forward(state)
        
        # Get best action
        best_action_idx = np.argmax(q_values)
        best_action = best_action_idx / (agent.action_dim - 1)
        
        # Extract state components
        inventory_norm = state[0]
        pipeline_norm = state[1]
        time_frac = state[2]
        
        # Convert to actual values
        inventory_level = inventory_norm * env.max_inventory
        pipeline_level = pipeline_norm * env.max_inventory
        
        policy_data.append({
            'inventory_level': inventory_level,
            'pipeline_level': pipeline_level,
            'time_fraction': time_frac,
            'action': best_action,
            'q_value': q_values[best_action_idx],
            'q_variance': np.var(q_values)
        })
    
    return pd.DataFrame(policy_data)

# Analyze the learned policy
policy_df = analyze_policy(agent, env, num_test_states=200)

print(f"Policy Analysis Results ({len(policy_df)} test states):")
print(f"Average Action (order ratio): {policy_df['action'].mean():.3f}")
print(f"Action Standard Deviation: {policy_df['action'].std():.3f}")
print(f"Average Q-Value: {policy_df['q_value'].mean():.2f}")
print(f"Q-Value Variance: {policy_df['q_variance'].mean():.2f}")

# Analyze action patterns by inventory level
low_inventory = policy_df[policy_df['inventory_level'] < 15000]
medium_inventory = policy_df[(policy_df['inventory_level'] >= 15000) & (policy_df['inventory_level'] < 25000)]
high_inventory = policy_df[policy_df['inventory_level'] >= 25000]

print(f"\nAction Patterns by Inventory Level:")
print(f"Low Inventory (<15k): Avg action = {low_inventory['action'].mean():.3f} ({len(low_inventory)} states)")
print(f"Medium Inventory (15k-25k): Avg action = {medium_inventory['action'].mean():.3f} ({len(medium_inventory)} states)")
print(f"High Inventory (>25k): Avg action = {high_inventory['action'].mean():.3f} ({len(high_inventory)} states)")

# Analyze action patterns by time within review period
early_time = policy_df[policy_df['time_fraction'] < 0.3]
mid_time = policy_df[(policy_df['time_fraction'] >= 0.3) & (policy_df['time_fraction'] < 0.7)]
late_time = policy_df[policy_df['time_fraction'] >= 0.7]

print(f"\nAction Patterns by Time in Review Period:")
print(f"Early (<30%): Avg action = {early_time['action'].mean():.3f} ({len(early_time)} states)")
print(f"Middle (30%-70%): Avg action = {mid_time['action'].mean():.3f} ({len(mid_time)} states)")
print(f"Late (>70%): Avg action = {late_time['action'].mean():.3f} ({len(late_time)} states)")

print("\nPolicy Insights:")
if low_inventory['action'].mean() > medium_inventory['action'].mean() > high_inventory['action'].mean():
    print("✓ Agent learns to order more when inventory is low (intuitive)")
else:
    print("⚠ Agent behavior may need more training")

if late_time['action'].mean() > early_time['action'].mean():
    print("✓ Agent learns to order more as review period approaches")
else:
    print("⚠ Agent doesn't show expected time-based ordering pattern")

In [ ]:
# Summary and Conclusions
print("=" * 70)
print("PERIODIC REVIEW BASE-STOCK POLICY - TIER 4 SUMMARY")
print("=" * 70)
print()
print("DEEP REINFORCEMENT LEARNING APPROACH:")
print("• Deep Q-Network with experience replay and target networks")
print("• State space: inventory, pipeline, time, demand history")
print("• Action space: continuous order quantity decisions")
print("• Training: 1000 episodes with epsilon-greedy exploration")
print("• Environment: non-stationary demand with trends and seasonality")
print()
print("TRAINING ACHIEVEMENTS:")
print(f"• Episodes Trained: {len(training_df)}")
print(f"• Cost Reduction: {cost_improvement:.1f}% during training")
print(f"• Service Improvement: {service_improvement:.1f}% during training")
print(f"• Final Epsilon: {final_performance['epsilon']:.3f}")
print(f"• Network Architecture: 4 layers, {agent.q_network.hidden_dim} hidden units")
print()
print("PERFORMANCE VS TRADITIONAL METHODS:")
print(f"• Traditional Cost: ${traditional_cost:.2f}/period")
print(f"• DQN Cost: ${test_stats['avg_cost']:.2f}/period")
print(f"• Cost Improvement: {cost_improvement_vs_traditional:.1f}%")
print()
print(f"• Traditional Service: {traditional_service:.3f} ({traditional_service*100:.1f}%)")
print(f"• DQN Service: {test_stats['avg_service']:.3f} ({test_stats['avg_service']*100:.1f}%)")
print(f"• Service Improvement: {service_improvement_vs_traditional:.1f}%")
print()
print("LEARNED POLICY CHARACTERISTICS:")
print(f"• Average Order Ratio: {policy_df['action'].mean():.3f}")
print(f"• Order Variability: {policy_df['action'].std():.3f}")
print(f"• Inventory-Responsive: {low_inventory['action'].mean():.3f} (low) vs {high_inventory['action'].mean():.3f} (high)")
print(f"• Time-Aware: {early_time['action'].mean():.3f} (early) vs {late_time['action'].mean():.3f} (late)")
print()
print("KEY ADVANTAGES OF DQN APPROACH:")
print("• Adapts to non-stationary demand patterns")
print("• Learns complex state-action relationships")
print("• Improves performance through experience")
print("• Handles uncertainty and volatility effectively")
print("• Provides continuous learning capability")
print()
print("PRACTICAL CONSIDERATIONS:")
print("• Requires training data and computational resources")
print("• Performance depends on hyperparameter tuning")
print("• Less interpretable than analytical methods")
print("• May need periodic retraining in changing environments")
print()
print("WHEN TO USE DEEP REINFORCEMENT LEARNING:")
print("• When demand patterns are non-stationary or complex")
print("• When you have abundant historical data")
print("• When adaptive performance provides competitive advantage")
print("• When traditional models fail to capture key patterns")
print("• When continuous improvement is valuable")
print()
print("COMPARISON SUMMARY:")
print("• Tier 1: Mathematical foundation with optimal static solutions")
print("• Tier 2: Practical heuristic with flexible review periods")
print("• Tier 3: Multi-objective optimization with strategic trade-offs")
print("• Tier 4: Adaptive learning with dynamic performance improvement")
print()
print("FINAL ASSESSMENT:")
if cost_improvement_vs_traditional > 0:
    print(f"✓ DQN achieves {cost_improvement_vs_traditional:.1f}% cost reduction")
if service_improvement_vs_traditional > 0:
    print(f"✓ DQN achieves {service_improvement_vs_traditional:.1f}% service improvement")
print("✓ Demonstrates value of adaptive learning in inventory management")
print("✓ Provides foundation for continuous improvement systems")